***

# **Congestion Data Formatting Script**

***


This file contains a script for formatting Annual Hours of Peak Hour Excessive Delay Per Capita for use of RITIS data. The prerequisites needed for this to work, is to manually download all of the PHED files for the given UZA's, and to store all of these files into one folder. In addition, all of these files should be named using the following example schematic: Annual Hours PHED Per Capita_3-7pm_Austin_TX. The only thing the user will be changing in the file name should be the city and the state abbreviation. The PHED files can be made via the [NPMRDS analytics tool](https://npmrds.ritis.org/analytics/my-dashboard/) and selecting the MAP-21 widget. Search for your UZA, select the Annual Hours of Peak Hour Excessive Delay Per Capita box, and then add all of your years. You will be redirected to see a chart for all of the years selected detailing PHED. You can save this data, in the top right of the panel widget. A good tip to know, is you are able to simply edit your already existing widget to swap out the UZA and the name of the file, rather than re-inputting all of the years again. Additionally, to get the Percent of Eligible Miles missing PHED, you need to make each UZA's dashboard to only feature the last year with full data. So for 2024, we use 2023 data. From there, you look at the bottom right of the chart produced and subtract that number from 100%. 

***

## **Congestion_1**

***

In [ ]:
import os
import pandas as pd
import re

# Plotting
import matplotlib.pyplot as plt 
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

# Set file paths

# Git
path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')

# AGOL Path for Pete
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'RTIS Data')

# Path with all of the PHED files named via the aforementioned schematic
path_congestion = os.path.join(path_sp, 'Data', 'Safe Equitable Resilient Infrastructure', 'Congestion')
path_phed = os.path.join(path_congestion, 'RTIS', 'PHED')
path_lottr = os.path.join(path_congestion, 'RTIS', 'LOTTR')

print(user)
print(path_git)

***

Congestion_1

***

In [ ]:
indicator_name = 'Congestion_1'

In [ ]:
# Importing the data to follow a naming pattern

df_list = []

# Iterate over every file in path
for filename in os.listdir(path_phed):
    if filename.endswith('.csv'):
        
        # Extract city name from the filename
        uza = filename.split('_')[2]
        
        # Load the file into a df
        file_path = os.path.join(path_phed, filename)
        df = pd.read_csv(file_path)
        
        # Adding new col w/ UZA name. This is for joining later
        df['UZA'] = uza

        df_list.append(df)

# Joining now
df_congestion1 = pd.concat(df_list, axis = 0, ignore_index=True)
df_congestion1['Month'] = pd.to_datetime(df_congestion1['Month']) #format = "%Y/%m"
df_congestion1['Month'] = df_congestion1['Month'].dt.to_period('M')
display(df_congestion1.head())

In [ ]:
df_plot = df_congestion1.copy()

df_plot['Year'] = df_plot['Month'].dt.year
df_plot = df_plot[df_plot['Year'] != 2011]
df_plot = df_plot.groupby(['Year', 'UZA']).sum(numeric_only = True)
df_plot = df_plot.reset_index()
df_plot = df_plot[['Year', 'UZA', 'PHED (hours)']]
# df_plot.to_csv(os.path.join(path_agol, indicator_name + '_UZA_RTIS.csv'), index = False)

fig = px.line(df_plot, x='Year', y='PHED (hours)', color='UZA', markers=True)
fig.update_layout(title = 'Total PHED (hours) by UZA')

# fig.write_html(os.path.join(path_congestion, indicator_name, 'plots', indicator_name + '_Total PHED by UZA_line.html'))

fig.show()

***

## **Congestion_3**

***

For Congestion_3, we only need data for SACOG counties. To do this, we use the same tool as in Congestion_1, but now we use the MPA for SACOG instead. Select the first three measures, and do the same as we did before. 

In [ ]:
# Loading the files

path_truck      = os.path.join(path_lottr, 'Truck Travel Time Reliability - Sacramento.csv')
path_interstate = os.path.join(path_lottr, 'Interstate Travel Time Reliability - Sacramento.csv')
path_noint     = os.path.join(path_lottr, 'Non-interstate NHS Travel Time Reliability - Sacramento.csv')

# reading files

df_truck = pd.read_csv(path_truck)
df_inter = pd.read_csv(path_interstate)
df_noint = pd.read_csv(path_noint)

# specifying LOTTR for int and non-int

df_inter.rename(columns={'LOTTR (%)': 'Interstate LOTTR (%)'    }, inplace=True)
df_noint.rename(columns={'LOTTR (%)': 'Non-Interstate LOTTR (%)'}, inplace=True)


# Merge
df_congestion3 = df_truck.merge(df_inter, on='Month', how='left').merge(df_noint, on='Month', how='left')
df_congestion3['Month'] = pd.to_datetime(df_congestion3['Month']) #format = "%Y/%m"
df_congestion3['Month'] = df_congestion3['Month'].dt.to_period('M')
df_congestion3 = df_congestion3.dropna()
df_congestion3 = df_congestion3.reset_index(drop = True)

# Get the averages per year now
display(df_congestion3)

In [ ]:
df_plot = df_congestion3.copy()

df_plot['Year'] = df_plot['Month'].dt.year
df_plot = df_plot[df_plot['Year'] != 2011]
df_plot = df_plot.groupby(['Year', 'UZA']).mean(numeric_only = True)
df_plot = df_plot.reset_index()


fig = px.line(df_plot, x='Year', y='PHED (hours)', color='UZA', markers=True)
fig.update_layout(title = 'Total PHED (hours) by UZA')

fig.write_html(os.path.join(path_congestion, indicator_name, 'plots', indicator_name + '_Total PHED by UZA_line.html'))

df_plot.columns = [col.lower() for col in df_plot.columns]
df_plot.columns = [re.sub('[\s+]', '_', col.strip()) for col in df_plot.columns]
df_plot.columns = [re.sub('\\?'  , '' , col.strip()) for col in df_plot.columns]
df_plot.to_csv(os.path.join(path_agol, indicator_name + '_UZA_RTIS.csv'), index = False)

fig.show()

***

## **Exports**

***

In [ ]:
# two exports for data folder | one in date time other in weighted average per year

# one in task 8 for average per year

# we do these exports for both congestion_1 and congestion_3

# Exports
indicator_name = 'Congestion'

# congestion_1
one_output_xlsx = [indicator_name, '_', '1', '.xlsx']
one_output_xlsx = "".join(one_output_xlsx)
one_output_csv = [indicator_name, '_', '1', '_UZA_RTIS', '.csv']
one_output_csv = "".join(one_output_csv)

# congestion_3
three_output_xlsx = [indicator_name, '_', '3', '.xlsx']
three_output_xlsx = "".join(three_output_xlsx)
three_output_csv = [indicator_name, '_', '3', '.csv']
three_output_csv = "".join(three_output_csv)

In [ ]:
# Set file path for exporting

path_out_one = os.path.join(path_congestion, 'Congestion_1')
path_out_three = os.path.join(path_congestion, 'Congestion_3')

# Congestion_1 Exports to SP
with pd.ExcelWriter(os.path.join(path_out_one, one_output_xlsx), engine='xlsxwriter') as writer:
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    congestion_1.to_excel(writer, index = False, sheet_name = 'UZA Long'       )
    congestion_1_wide.to_excel(writer, index = False, sheet_name = 'UZA Wide'  )

# Congestion_1 Exports AGOL
congestion_1_year.to_csv(os.path.join(path_agol, one_output_csv), index = False)
congestion_1_wide_year.to_csv(os.path.join(path_agol, 'Congestion_1 Wide.csv'))

# Congestion_3 Exports
with pd.ExcelWriter(os.path.join(path_out_three, three_output_xlsx), engine='xlsxwriter') as writer:
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    congestion_3.to_excel(writer, index = False, sheet_name = 'SACOG'       )

congestion_3_year.to_csv(os.path.join(path_agol, three_output_csv), index = False)

print('Congestion_1 SP Files Exported Here: ' + path_out_one)
print('Congestion_3 Files Exported Here: '   + path_out_three)
print('Congestion_1 & Congestion_3 AGOL Files Exported Here: ' + path_agol)